# Notebook 3 – Train, Validation & Test

In [1]:
import pandas as pd

df = pd.read_csv("data.csv", encoding="latin1")
df = df.dropna(subset=["Description"]).copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C").astype(int)

df["AbsQuantity"] = df["Quantity"].abs()
df["Month"] = df["InvoiceDate"].dt.month
df["IsInternational"] = (df["Country"] != "United Kingdom").astype(int)

feature_cols = ["AbsQuantity", "UnitPrice", "Month", "IsInternational"]
X = df[feature_cols].fillna(0)
y = df["IsCancelled"]
X.shape

(540455, 4)

## Training Dataset

**What it is:** the data the model actually learns from, calling
`.fit()` on this set adjusts the model's parameters.

## Validation Dataset

**What it is:** a separate set used *during development* to tune
hyperparameters and compare different models, without touching the final
test set. In Notebook 2, `GridSearchCV`'s internal cross-validation folds
served this role.

## Test Dataset

**What it is:** touched exactly once, at the very end, to report honest,
unbiased performance. If tuning decisions were influenced by test set
results, the test set has effectively become a second validation set,
and the reported score is no longer trustworthy.

## Train/Test Split

The simplest split, useful when you don't need extensive hyperparameter
tuning, or when cross-validation (below) will handle validation
internally.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (432364, 4) | Test: (108091, 4)


## Train/Validation/Test Split

For more careful model development, a three-way split keeps tuning
completely separate from final evaluation.

In [3]:
train_val_X, test_X, train_val_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_X, val_X, train_y, val_y = train_test_split(train_val_X, train_val_y, test_size=0.25, random_state=42, stratify=train_val_y)

print("Train:", train_X.shape, "| Validation:", val_X.shape, "| Test:", test_X.shape)

Train: (324273, 4) | Validation: (108091, 4) | Test: (108091, 4)


## Cross Validation

**What it does:** instead of a single fixed validation split, the
training data is divided into several "folds." The model trains on all
but one fold, validates on the held-out fold, and this repeats until
every fold has served as validation once. Results are averaged, giving a
more stable performance estimate than one lucky (or unlucky) single
split.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight="balanced")

cv_scores = cross_val_score(model, train_val_X, train_val_y, cv=5, scoring="roc_auc")
print("ROC AUC per fold:", cv_scores.round(3))
print("Mean ROC AUC:", round(cv_scores.mean(), 3), "| Std:", round(cv_scores.std(), 3))

## K-Fold Cross Validation

The general technique above is called K-Fold Cross Validation, where K is
the number of folds (we used K=5). Larger K means more folds, more
training runs, and a more stable estimate, but at a higher computational
cost.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_scores = cross_val_score(model, train_val_X, train_val_y, cv=kfold, scoring="roc_auc")
print("Plain K-Fold ROC AUC per fold:", kfold_scores.round(3))

## Stratified K-Fold

**What it does:** the same K-Fold idea, but ensures each fold preserves
the original class proportions. Given our real class imbalance
(`IsCancelled` is roughly 82%/18%), plain K-Fold shuffling could
accidentally create folds with very different cancellation rates by
chance. Stratified K-Fold prevents that.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(model, train_val_X, train_val_y, cv=skf, scoring="roc_auc")
print("Stratified K-Fold ROC AUC per fold:", skf_scores.round(3))

# Confirm each fold keeps a similar class balance
for i, (train_idx, val_idx) in enumerate(skf.split(train_val_X, train_val_y)):
    fold_balance = train_val_y.iloc[val_idx].mean()
    print(f"Fold {i+1} cancellation rate: {fold_balance:.3f}")

## Random State

**What it does:** fixes the randomness used when splitting or shuffling
data, so results are reproducible. Every split and cross-validation call
above used `random_state=42`, running this notebook again will produce
the exact same splits and, therefore, comparable results.

In [ ]:
# Same random_state produces identical splits every time
split_a, _ = train_test_split(X, test_size=0.2, random_state=42)
split_b, _ = train_test_split(X, test_size=0.2, random_state=42)
print("Identical splits confirmed:", split_a.index.equals(split_b.index))

## Data Leakage

Already demonstrated directly in Notebook 1: using raw `Quantity`
(negative exactly when cancelled) produced a fake 1.00 accuracy. That's
leakage, the model wasn't learning a real pattern, it was reading a
disguised version of the answer. The fix, `AbsQuantity`, is why every
feature set in this notebook uses it instead.

## Overfitting

**What it is:** a model that performs very well on training data but
much worse on unseen data, it has memorized noise and specifics of the
training set rather than learning a generalizable pattern.

**A note on measuring this honestly:** with our real class imbalance
(roughly 82%/18%), plain accuracy can barely move even between a
severely overfit model and an underfit one, a model can score ~98%
accuracy just by leaning on the majority class, regardless of how much it
has actually overfit. ROC AUC, which depends on how well the model ranks
predicted probabilities rather than raw correct/incorrect counts, is a
much more honest lens here. We use it below instead of accuracy.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

overfit_model = DecisionTreeClassifier(max_depth=None, random_state=42)  # no depth limit
overfit_model.fit(train_X, train_y)

train_auc = roc_auc_score(train_y, overfit_model.predict_proba(train_X)[:, 1])
val_auc = roc_auc_score(val_y, overfit_model.predict_proba(val_X)[:, 1])
print("Overfit model, Training ROC AUC:", round(train_auc, 3))
print("Overfit model, Validation ROC AUC:", round(val_auc, 3))
print("Gap:", round(train_auc - val_auc, 3), "-> a large gap signals overfitting")

## Underfitting

**What it is:** a model too simple to capture the real pattern in the
data, performing poorly on both training and validation data alike.

In [ ]:
underfit_model = DecisionTreeClassifier(max_depth=1, random_state=42)  # extremely shallow
underfit_model.fit(train_X, train_y)

train_auc_under = roc_auc_score(train_y, underfit_model.predict_proba(train_X)[:, 1])
val_auc_under = roc_auc_score(val_y, underfit_model.predict_proba(val_X)[:, 1])
print("Underfit model, Training ROC AUC:", round(train_auc_under, 3))
print("Underfit model, Validation ROC AUC:", round(val_auc_under, 3))
print("Both scores sit near 0.5 (random guessing) -> underfitting, not enough model complexity to learn the pattern")

## Comparing All Three: Underfit, Balanced, Overfit

In [ ]:
balanced_model = DecisionTreeClassifier(max_depth=6, random_state=42)
balanced_model.fit(train_X, train_y)

def auc_pair(m):
    return (
        roc_auc_score(train_y, m.predict_proba(train_X)[:, 1]),
        roc_auc_score(val_y, m.predict_proba(val_X)[:, 1])
    )

underfit_scores = auc_pair(underfit_model)
balanced_scores = auc_pair(balanced_model)
overfit_scores = auc_pair(overfit_model)

comparison = pd.DataFrame({
    "Model": ["Underfit (depth=1)", "Balanced (depth=6)", "Overfit (no depth limit)"],
    "Train ROC AUC": [underfit_scores[0], balanced_scores[0], overfit_scores[0]],
    "Validation ROC AUC": [underfit_scores[1], balanced_scores[1], overfit_scores[1]]
})
comparison["Gap"] = comparison["Train ROC AUC"] - comparison["Validation ROC AUC"]
comparison.round(3)

## Why the Test Dataset Should Never Be Used for Model Tuning

This is the central rule this notebook builds toward, and the reasoning
follows directly from everything demonstrated above:

* **Cross-validation and the validation set exist specifically so tuning
  never needs to touch the test set.** Every hyperparameter decision in
  Notebook 2 (`GridSearchCV`) and every fold comparison above used
  training/validation data only, the test set was never part of that
  process.

* **If the test set is used for tuning, it silently becomes a validation
  set.** Once you've adjusted `max_depth`, tried different features, or
  picked a model based on test set performance, that test score no longer
  reflects performance on genuinely unseen data, it reflects performance
  on data the tuning process was indirectly shaped around.

* **The overfitting demonstration above shows exactly why this matters.**
  Our overfit model scored much higher ROC AUC on training data than on
  validation data, a real gap, while accuracy alone barely moved due to
  class imbalance. If we'd used the *test* set instead of a validation
  set to notice this gap and adjust `max_depth` accordingly, we would
  have tuned the model to fit the test set's specific quirks too, and the
  final "test score" we report would be optimistic, not honest.

* **A test score should answer one question, and only be asked once:**
  "how will this final, already-decided model perform on data it has
  truly never influenced in any way?" The moment it's used more than
  once, or used to make a decision, it can no longer answer that question
  reliably.

In [ ]:
# Correct final step: evaluate the tuned model from Notebook 2 on the test set ONCE, at the very end
final_model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight="balanced")
final_model.fit(train_val_X, train_val_y)  # trained on train+validation combined, tuning already finished

from sklearn.metrics import roc_auc_score
final_test_score = roc_auc_score(test_y, final_model.predict_proba(test_X)[:, 1])
print("Final, one-time test set ROC AUC:", round(final_test_score, 3))
print("This number is now final. No further tuning decisions should be made using it.")